In [2]:
import sys
!{sys.executable} -m pip install -U openai


'g:\Sanket' is not recognized as an internal or external command,
operable program or batch file.


## Using OPENAI And GROQ

In [18]:
from dataclasses import dataclass
from openai import OpenAI
import os

@dataclass(frozen=True)
class Provider:
    """One provider to reliably route requests accross all inference providers """

    name: str
    env_var: str
    is_free: bool
    base_url: str | None
    model: str

PROVIDERS = [
    # Provider("OpenAI", "OPENAI_API_KEY", True, None, "gpt-4o-mini"),
    # Provider("Groq", "GROQ_API_KEY", True, "https://api.groq.com/openai/v1", "openai/gpt-oss-120b"),
    Provider("OpenRouter", "OPEN_ROUTER_API_KEY", True, "https://openrouter.ai/api/v1", "openai/gpt-oss-120b")
]


def select_provider() -> Provider:
    for provider in PROVIDERS:
        if os.getenv(provider.env_var):
            return provider
    
    expected = ", ".join(p.env_var for p in PROVIDERS)
    raise RuntimeError(f"No provider key set, add one of {expected} to your environment variables")

def build_client(provider: Provider) -> OpenAI:

    api_key = os.getenv(provider.env_var)
    if provider.base_url is None:
        return OpenAI(api_key=api_key)
    
    return OpenAI(
        api_key=api_key,
        base_url=provider.base_url,
    )


def have_any_key() -> bool:
    return any(os.getenv(p.env_var) for p in PROVIDERS)

print("Found a provider key." if have_any_key() else "No provider key found.")

Found a provider key.


In [19]:
def llm_reply(prompt: str) -> str:
    provider = select_provider()
    print(f"Using {provider.name} provider")
    client = build_client(provider)
    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=200,
        messages=[{
            "role": "user",
            "content": prompt,
        }]
    )

    return result.choices[0].message.content

In [20]:
prompt = "Who was the PM of UK before 2020 ? Answer in a single sentence"

try:
    print(llm_reply(prompt))
except Exception as e:
    print(f"Error: {e}")

Using OpenRouter provider
The Prime Minister of the United Kingdom immediately preceding 2020 was Boris Johnson, who took office on 24 July 2019.


HW: Modify the implementation of providers to add a OPEN Router provider as well.

In [ ]:
# Completed above

In [21]:
def chat_reply(messages: list[dict]) -> str:
    provider = select_provider()

    client = build_client(provider)

    result = client.chat.completions.create(
        model=provider.model,
        max_tokens=1000,
        messages=messages,
    )

    return result.choices[0].message.content

In [22]:
conversation = []

In [23]:
conversation.append({"role": "user", "content": "What is the capital of France?"})

In [24]:
reply_from_llm = chat_reply(conversation)
print(reply_from_llm)

conversation.append({
    "role": "assistant",
    "content": reply_from_llm,
})

The capital of France is **Paris**.


In [25]:
conversation

[{'role': 'user', 'content': 'What is the capital of France?'},
 {'role': 'assistant', 'content': 'The capital of France is **Paris**.'}]

In [26]:
conversation.append({
    "role": "user",
    "content": "what is their GDP ? ",
})

In [27]:
conversation

[{'role': 'user', 'content': 'What is the capital of France?'},
 {'role': 'assistant', 'content': 'The capital of France is **Paris**.'},
 {'role': 'user', 'content': 'what is their GDP ? '}]

In [28]:
llm_reply = chat_reply(conversation)
print(llm_reply)

**France’s Gross Domestic Product (GDP)**  

| Year | Nominal GDP (USD) | Nominal GDP (EUR) | GDP per capita (USD) | Source |
|------|-------------------|-------------------|----------------------|--------|
| **2023** | **≈ $2.94 trillion** | **≈ €2.73 trillion** | **≈ $43,500** | International Monetary Fund (IMF) World Economic Outlook, April 2024; Eurostat “GDP and main aggregates” (2024 release) |
| **2022** | **≈ $2.78 trillion** | **≈ €2.60 trillion** | **≈ $41,200** | World Bank “World Development Indicators” (accessed June 2024) |
| **2021** | **≈ $2.71 trillion** | **≈ €2.55 trillion** | **≈ $40,300** | OECD “National Accounts” (2024 update) |

### Key points

1. **Nominal vs. real GDP**  
   - The figures above are **nominal** (current‑price) GDP, expressed in both U.S. dollars and euros.  
   - Real (inflation‑adjusted) GDP growth for 2023 was about **+1.6 %** year‑on‑year, according to the IMF.

2. **GDP composition (2023)**  
   - **Services** dominate the French economy, a

# Key takeaways
- A bare model is a one-shot function: text in, text out, no memory.
- A chatbot = model + a transcript list of {"role", "content"} dicts. Sending that list back each turn is memory.
- A real call is still text-in / text-out, just wrapped: a messages list goes in, choices[0].message.content comes out (OpenAI dialect).
- All three providers speak the OpenAI dialect — Groq & OpenRouter just reuse the openai SDK with a different base_url, so one call path covers them all.